# 01. Data Preparation

**Paper section:** §3 Data, §3.1 Corpora and preprocessing.
**What it computes:** Loads ORACC Akkadian / Sumerian and the curated Elamite lemma base + Nasu corpus, harmonizes POS tags, and converts every form to Unicode cuneiform. Caches the unified token-level dataframes used by every downstream notebook.
**Inputs:** `alltexts_AKK.csv`, `alltexts_SUX.csv` (produced from the Zenodo ORACC dump by `scripts/filter_oracc_data.py`), `Elamite_Lemma-base-draft.xlsx`, `UnTN-Nasu texts Word-level.csv`, and the two `unmatchednew*.csv` manual sign corrections.
**Outputs:** `outputs/dataset_summary.json` (token / type counts per language). The Akkadian / Sumerian / Elamite dataframes returned by `load_corpora()` are the canonical inputs for notebooks 02-08.
**Expected runtime (CPU baseline):** ~3 min on the curated sample; ~15 min once ORACC `alltexts_*.csv` are present.

All randomness uses `SEED = 42`.

## Setup
This notebook uses shared utilities from `_setup.py`:
- `load_corpora()`: returns dict of dataframes per language
- `POS_HARMONIZATION`: dict mapping raw POS tags to unified tagset
- `BASE_PATH`: data directory (set this for your environment)

```python
from _setup import load_corpora, POS_HARMONIZATION, BASE_PATH
```


In [ ]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)
try:
    import torch
    torch.manual_seed(42)
except ImportError:
    pass

from _setup import load_corpora, POS_HARMONIZATION, BASE_PATH, set_seeds
set_seeds(42)


## Load all corpora

This exercises `_setup.load_corpora`. With no data present it prints `[warn]` lines and returns an empty dict — drop the curated files into `BASE_PATH` to populate.


In [ ]:
corpora = load_corpora(BASE_PATH)
for lang in ('akk', 'sux', 'elx'):
    if lang in corpora:
        df = corpora[lang]
        print(f'{lang}: {len(df):,} tokens, {df["text_id"].nunique():,} texts, {df["form_latin"].nunique():,} types')
    else:
        print(f'{lang}: not loaded')


## Persist a small dataset summary


In [ ]:
import json, os
os.makedirs('../outputs', exist_ok=True)
summary = {}
for lang in ('akk', 'sux', 'elx'):
    if lang in corpora:
        df = corpora[lang]
        summary[lang] = {
            'tokens': int(len(df)),
            'texts':  int(df['text_id'].nunique()),
            'types_latin':   int(df['form_latin'].nunique()),
            'types_unicode': int(df['form_unicode'].nunique()) if 'form_unicode' in df.columns else None,
        }
with open('../outputs/dataset_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
summary


## Reference: how to produce `alltexts_*.csv` from ORACC

The Akkadian and Sumerian token files are not redistributed with this repo — they're derived from the public ORACC dump on Zenodo (`finaldf.csv`, ~1.5 GB).

```bash
# from the directory containing finaldf.csv:
python scripts/filter_oracc_data.py
# produces alltexts_AKK.csv and alltexts_SUX.csv
```
Move those two files into `BASE_PATH`.

**ORACC dump:** <https://zenodo.org/records/10794626/files/finaldf.csv>
